# 🏛️ ĐỒ ÁN MÔN HỌC DAT301m - STAGE 2: GEOMETRIC NORMALIZATION & BACKGROUND WHITENING
## Căn giữa Bounding Box, Chuẩn hóa Tỷ lệ và Tẩy trắng Nền giấy Scan

* **Vấn đề cốt tử:** Ảnh scan có nét chữ chỉ chiếm 15-30% diện tích và nằm lệch góc. Khi downsample trực tiếp 8 lần về $(32, 32)$, nét vẽ co lại còn vệt mờ 2-3 pixel khiến VGG16 không thể phân giải. Nền scan xám gây phân tán bộ lọc.
* **Can thiệp Data:**
  1. Bounding-Box Detection: Dò nét chữ, crop sát, đệm viền 15%, căn giữa khung vuông. Chữ chiếm **75% - 80%** diện tích canvas.
  2. Background Whitening: Toàn bộ pixel nền > 215 đưa về trắng tuyệt đối (255) và tăng tương phản nét bút.
  3. Chuẩn hóa đồng bộ cho cả tập `label_book`.


In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("🚀 Đang chạy trên Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Đang chạy trên Local Machine")

if IN_COLAB:
    if os.path.exists("/content/deeplearningai-data-centric-competition"):
        os.chdir("/content/deeplearningai-data-centric-competition")
    elif not os.path.exists("data"):
        os.system("git clone https://github.com/Nov05/deeplearningai-data-centric-competition.git")
        if os.path.exists("deeplearningai-data-centric-competition"):
            os.chdir("deeplearningai-data-centric-competition")

# Giải nén data và label_book nếu chưa có
import tarfile
if not os.path.exists("data") and os.path.exists("original/data.tar.gz"):
    with tarfile.open("original/data.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu data thành công.")

if not os.path.exists("label_book") and os.path.exists("original/label_book.tar.gz"):
    with tarfile.open("original/label_book.tar.gz", "r:gz") as tar:
        tar.extractall(".")
    print("✅ Đã giải nén dữ liệu label_book thành công.")

import glob
raw_count = len(glob.glob("data/**/*.png", recursive=True))
print(f"✅ Dữ liệu gốc: Tìm thấy {raw_count} ảnh.")
assert raw_count > 0, "LỖI: Chưa có ảnh trong thư mục 'data/'!"


🚀 Đang chạy trên Google Colab


/tmp/ipykernel_531/3864034174.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


✅ Đã giải nén dữ liệu data thành công.
✅ Dữ liệu gốc: Tìm thấy 2880 ảnh.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import hashlib
import shutil
from PIL import Image

IMAGE_SIZE = (32, 32)
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASS_NAMES = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
RANDOM_SEED = 123
EPOCHS = 15

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU ({gpus[0].name}) -> Tốc độ huấn luyện cực nhanh!")
else:
    print("⚠️ Thiết bị hiện tại: CPU (Khuyên dùng: Trên Colab vào Menu Runtime -> Change runtime type -> Chọn T4 GPU)")

def build_fixed_vgg16_model():
    """
    KIẾN TRÚC VGG16 TỰ XÂY DỰNG TỪ ĐẦU (FROM SCRATCH) THEO YÊU CẦU GIẢNG VIÊN:
    - KHÔNG dùng thư viện có sẵn tf.keras.applications.VGG16
    - KHÔNG dùng pretrained weights ImageNet
    - Tự code tay đúng 13 lớp Conv2D + 5 lớp MaxPooling2D + 3 lớp Dense = 16 Layers VGG16
    - Khởi tạo He-Normal, cố định bất biến qua mọi Stage
    """
    tf.random.set_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name="input_image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescaling")(inputs)

    # Block 1 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c1")(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b1_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b1_pool")(x)

    # Block 2 (2 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c1")(x)
    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b2_c2")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b2_pool")(x)

    # Block 3 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c1")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c2")(x)
    x = tf.keras.layers.Conv2D(256, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b3_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b3_pool")(x)

    # Block 4 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b4_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b4_pool")(x)

    # Block 5 (3 Conv + MaxPool)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c1")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c2")(x)
    x = tf.keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu", kernel_initializer="he_normal", name="b5_c3")(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), strides=(2, 2), name="b5_pool")(x)

    # Classifier Head (13 Conv + 3 Dense = 16 Layers VGG16)
    x = tf.keras.layers.Flatten(name="flatten")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc1")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout1")(x)
    x = tf.keras.layers.Dense(512, activation="relu", kernel_initializer="he_normal", name="fc2")(x)
    x = tf.keras.layers.Dropout(0.5, name="dropout2")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Fixed_VGG16_Scratch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).")


TensorFlow Version: 2.21.0
🚀 Thiết bị tăng tốc: ĐÃ BẬT GPU (/physical_device:GPU:0) -> Tốc độ huấn luyện cực nhanh!
✅ Đã khởi tạo cấu trúc Fixed VGG16 Scratch thành công (15.245.130 tham số).


In [ ]:
def preprocess_and_center_numeral(img_pil, target_size=(32, 32), pad_ratio=0.15):
    gray = img_pil.convert("L")
    arr = np.array(gray)
    arr[arr > 215] = 255 # Tẩy trắng nền scan
    ys, xs = np.where(arr < 215)
    if len(ys) == 0:
        return None
    ymin, ymax = ys.min(), ys.max()
    xmin, xmax = xs.min(), xs.max()
    h = ymax - ymin + 1
    w = xmax - xmin + 1
    max_dim = max(h, w)
    pad = int(max_dim * pad_ratio)
    total_size = max_dim + 2 * pad
    canvas = np.ones((total_size, total_size), dtype=np.uint8) * 255
    crop = arr[ymin:ymax+1, xmin:xmax+1]
    canvas[(total_size-h)//2:(total_size-h)//2+h, (total_size-w)//2:(total_size-w)//2+w] = crop
    canvas[canvas < 120] = (canvas[canvas < 120] * 0.7).astype(np.uint8) # Boost tương phản
    res = Image.fromarray(canvas).resize(target_size, Image.Resampling.BILINEAR)
    return res.convert("RGB")

def ensure_clean_label_book():
    CLEAN_LB_DIR = "clean_label_book"
    if not os.path.exists(CLEAN_LB_DIR) or len(glob.glob(f"{CLEAN_LB_DIR}/*/*.png")) == 0:
        os.makedirs(CLEAN_LB_DIR, exist_ok=True)
        for c in CLASS_NAMES:
            os.makedirs(os.path.join(CLEAN_LB_DIR, c), exist_ok=True)
            for f in glob.glob(f"label_book/{c}/*.png"):
                proc = preprocess_and_center_numeral(Image.open(f), target_size=IMAGE_SIZE)
                if proc is not None:
                    proc.save(os.path.join(CLEAN_LB_DIR, c, os.path.basename(f)))
        print("✅ Đã tự động khởi tạo clean_label_book chuẩn hóa.")


## 1. CHUẨN HÓA TOÀN BỘ DỮ LIỆU SẠCH VÀ LABEL BOOK


In [ ]:
STAGE2_DIR = "stage2_geometric_data"
if os.path.exists(STAGE2_DIR):
    shutil.rmtree(STAGE2_DIR)
os.makedirs(STAGE2_DIR, exist_ok=True)

ensure_clean_label_book()

for c in CLASS_NAMES:
    c_dest = os.path.join(STAGE2_DIR, c)
    os.makedirs(c_dest, exist_ok=True)
    raw_files = glob.glob(f"data/train/{c}/*.png") + glob.glob(f"data/val/{c}/*.png")
    seen_hashes = set()
    idx = 0
    for f in raw_files:
        if os.path.basename(f) in {"aac26b40-ce5d-11eb-b317-38f9d35ea60f.png", "b0a13faa-ce5d-11eb-b317-38f9d35ea60f.png"}:
            continue
        with open(f, "rb") as fp:
            h = hashlib.md5(fp.read()).hexdigest()
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
        proc = preprocess_and_center_numeral(Image.open(f), target_size=IMAGE_SIZE)
        if proc is not None:
            proc.save(os.path.join(c_dest, f"geo_{idx}.png"))
            idx += 1

print(f"✅ Đã chuẩn hóa toàn bộ hình học ảnh. Số ảnh: {len(glob.glob(f'{STAGE2_DIR}/*/*.png'))}")


✅ Đã tự động khởi tạo clean_label_book chuẩn hóa.
✅ Đã chuẩn hóa toàn bộ hình học ảnh. Số ảnh: 2829


## 2. HUẤN LUYỆN LẠI VGG16 SCRATCH TRÊN DỮ LIỆU ĐÃ CHUẨN HÓA HÌNH HỌC


In [ ]:
train_ds_s2 = tf.keras.preprocessing.image_dataset_from_directory(
    STAGE2_DIR, validation_split=0.2, subset="training", seed=RANDOM_SEED,
    labels="inferred", label_mode="categorical", class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

val_ds_s2 = tf.keras.preprocessing.image_dataset_from_directory(
    STAGE2_DIR, validation_split=0.2, subset="validation", seed=RANDOM_SEED,
    labels="inferred", label_mode="categorical", class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

clean_lb_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "clean_label_book", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, shuffle=False,
    batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

model_s2 = build_fixed_vgg16_model()
history_s2 = model_s2.fit(train_ds_s2, validation_data=val_ds_s2, epochs=EPOCHS, verbose=1)

val_loss_s2, val_acc_s2 = model_s2.evaluate(val_ds_s2, verbose=0)
lb_loss_s2, lb_acc_s2 = model_s2.evaluate(clean_lb_ds, verbose=0)

print("="*60)
print(f"📊 KẾT QUẢ STAGE 2 (GEOMETRIC & BACKGROUND WHITENING):")
print(f"  • Validation Accuracy: {val_acc_s2*100:.2f}% | Loss: {val_loss_s2:.4f}")
print(f"  • Label Book Accuracy: {lb_acc_s2*100:.2f}% | Loss: {lb_loss_s2:.4f}")
print("="*60)


Found 2829 files belonging to 10 classes.
Using 2264 files for training.
Found 2829 files belonging to 10 classes.
Using 565 files for validation.
Found 52 files belonging to 10 classes.
Epoch 1/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 54s 432ms/step - accuracy: 0.1131 - loss: 2.3399 - val_accuracy: 0.1204 - val_loss: 2.2925
Epoch 2/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.1237 - loss: 2.2961 - val_accuracy: 0.1363 - val_loss: 2.2832
Epoch 3/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.1868 - loss: 2.2098 - val_accuracy: 0.1664 - val_loss: 2.1602
Epoch 4/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.2566 - loss: 1.9963 - val_accuracy: 0.3894 - val_loss: 1.7020
Epoch 5/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.3843 - loss: 1.6900 - val_accuracy: 0.5947 - val_loss: 1.2840
Epoch 6/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.5486 - loss: 1.2410 - val_accuracy: 0.6460 - val_loss: 1.0613
Epoch 7/15
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/ste